# treewidth_cut チュートリアル

`treewidth_cut.py` は、量子回路の **インタラクショングラフ** を min-fill ヒューリスティックで解析し、
カットすると最もハードウェアゲート数 (ECR) を削減できる 2-qubit ゲートを提案するライブラリです。

## 目次

1. **高レベル API** (`find_optimal_cuts`) — ワンライナーでカット位置を選択
2. **CutResult の活用** — 結果の確認とカット済み回路の取得
3. **低レベル API** — 段階的な解析と可視化
4. **他の戦略との比較** (naive / random)
5. **K=2 カット** — 2箇所同時カット
6. パラメータの説明
7. **統合 API** (`execute_subcircuits`) — `DataClassSubQCParams` → カット → `DataClassSubQCRes`
8. **QPD ゲートカットと分布再構成** — 確率分布の再構成

## 0. セットアップ

In [ ]:
import sys
sys.path.insert(0, "/Users/ebihana/Project/NEDO2/")

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import logging

# stevedore の警告を抑制
logging.getLogger("stevedore").setLevel(logging.CRITICAL)

from qiskit import transpile
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

from circuit_preprocesser.treewidth_cut import (
    find_optimal_cuts,
    QCGraph,
    build_edge_map,
    min_fill_trace,
    score_edges_from_trace,
    cut_gate_by_index,
    evaluate_cost,
    CutResult,
)
from circuit_preprocesser.utils import create_multi_cx_qc

backend = FakeSherbrooke()

## 1. 高レベル API: `find_optimal_cuts`

最も簡単な使い方です。量子回路とバックエンドを渡すだけで、最適なカット位置を提案してくれます。

In [ ]:
# ランダムな 30-qubit, 40-CX 回路を作成
qc = create_multi_cx_qc(n_qubit=30, n_cx=40, seed=2)
print(f"回路: {qc.num_qubits} qubits, {len(qc.data)} gates")
qc.draw("mpl", fold=80)

In [ ]:
# ワンライナーで最適カット位置を検索
result = find_optimal_cuts(qc, backend, max_cuts=1)

print(f"カット対象エッジ:   {result.cut_edges}")
print(f"ゲートインデックス: {result.gate_indices}")
print(f"ECR (カット前):     {result.ecr_before}")
print(f"ECR (カット後):     {result.ecr_after}")
print(f"ECR 削減数:         {result.ecr_reduction}")
print(f"削減率:             {result.reduction_ratio:.1%}")
print(f"Treewidth 上界:     {result.treewidth_upper}")

## 2. CutResult の活用

`CutResult.apply()` でカット済み回路を取得できます。

In [ ]:
# カット済み回路を取得
qc_cut = result.apply(qc)
print(f"元の回路: {len(qc.data)} gates")
print(f"カット後: {len(qc_cut.data)} gates (ゲート {result.gate_indices} を削除)")

# トランスパイル後の ECR 数を確認
qc_t = transpile(qc, backend=backend, layout_method="sabre", seed_transpiler=42)
qc_cut_t = transpile(qc_cut, backend=backend, layout_method="sabre", seed_transpiler=42)

print(f"\nトランスパイル後 ECR:")
print(f"  元の回路: {qc_t.count_ops().get('ecr', 0)}")
print(f"  カット後: {qc_cut_t.count_ops().get('ecr', 0)}")

## 3. 低レベル API: 段階的な解析

内部で何が起きているかを段階的に見ていきます。

### 3.1 インタラクショングラフの構築

In [ ]:
# QCGraph: 量子回路から 2-qubit ゲートのインタラクショングラフを構築
qcg = QCGraph(qc)

print(f"ノード数: {len(qcg.node_lst)}")
print(f"エッジ数: {len(qcg.edge_lst)}")
print(f"エッジ (重み付き): {qcg.weighted_edge_lst[:5]} ...")

# グラフの可視化
fig, ax = plt.subplots(figsize=(8, 8))
pos = nx.circular_layout(qcg.G)
nx.draw_networkx_nodes(qcg.G, pos=pos, node_color="green", node_size=400, ax=ax)
nx.draw_networkx_edges(qcg.G, pos=pos, edge_color="black", ax=ax)
nx.draw_networkx_labels(qcg.G, pos=pos, font_color="white", ax=ax)
edge_labels = nx.get_edge_attributes(qcg.G, "weight")
nx.draw_networkx_edge_labels(qcg.G, pos=pos, edge_labels=edge_labels, ax=ax)
ax.set_title("Interaction Graph (weight = 2-qubit gate count)")
ax.axis("off")
plt.tight_layout()
plt.show()

### 3.2 エッジマップの構築

`build_edge_map` は、各エッジの出現頻度と、対応するゲートインデックスのリストを返します。

In [ ]:
w, edge_to_gate_indices = build_edge_map(qc)

print("エッジ頻度 (w):")
for edge, freq in sorted(w.items(), key=lambda x: -x[1])[:10]:
    gate_idxs = edge_to_gate_indices[edge]
    print(f"  {edge}: freq={freq}, gate_indices={gate_idxs}")

### 3.3 Min-fill トレース

`min_fill_trace` はグラフの消去順序を計算し、各ステップの bag サイズや fill-edge 情報を記録します。
Treewidth の上界 = max(bag_size - 1) です。

In [ ]:
order, steps, tw_upper = min_fill_trace(qcg.G)

print(f"Treewidth 上界: {tw_upper}")
print(f"消去順序: {order[:10]} ...")
print(f"\n各ステップ (bag_size >= 3 のもの):")
for s in steps:
    if s["bag_size"] >= 3:
        print(f"  v={s['v']:2d}, bag_size={s['bag_size']}, "
              f"fill_count={s['fill_count']}, fill_edges={s['fill_edges']}")

### 3.4 エッジスコアリング

fill-edge が多く発生するステップに関与するエッジほど高スコアになります。
高スコアのエッジをカットすると、treewidth が下がり、トランスパイル後のゲート数削減が期待できます。

In [ ]:
ranked = score_edges_from_trace(
    steps, qcg.G, w,
    alpha_bag=1.0, beta_fill=1.0, use_freq=True,
    topk=10,
)

print("エッジスコア (上位10):")
for edge, sc in ranked:
    has_gate = "Yes" if edge_to_gate_indices.get(edge) else "No"
    print(f"  {edge}: score={sc:.2f}, cuttable={has_gate}")

### 3.5 スコア付きグラフの可視化

エッジの色でスコアの高低を表示します。

In [ ]:
# スコア辞書を構築
score_dict = {e: sc for e, sc in ranked}
max_score = max(score_dict.values()) if score_dict else 1.0

fig, ax = plt.subplots(figsize=(8, 8))
pos = nx.circular_layout(qcg.G)

# ノード描画
nx.draw_networkx_nodes(qcg.G, pos=pos, node_color="green", node_size=400, ax=ax)
nx.draw_networkx_labels(qcg.G, pos=pos, font_color="white", ax=ax)

# エッジをスコアに応じて色分け
for u, v in qcg.edge_lst:
    e = (min(u, v), max(u, v))
    sc = score_dict.get(e, 0.0)
    color = plt.cm.Reds(sc / max_score) if sc > 0 else "lightgray"
    width = 1 + 3 * (sc / max_score) if sc > 0 else 0.5
    nx.draw_networkx_edges(qcg.G, pos=pos, edgelist=[(u, v)],
                           edge_color=[color], width=width, ax=ax)

# カット対象エッジをハイライト
for edge in result.cut_edges:
    nx.draw_networkx_edges(qcg.G, pos=pos, edgelist=[edge],
                           edge_color="blue", width=4, style="dashed", ax=ax)

ax.set_title("Edge Scores (red=high score, blue dashed=selected cut)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. 他の戦略との比較

提案手法 (treewidth-based) と、naive (先頭ゲート) / random の比較を行います。

In [ ]:
import random

n_qubit_range = range(10, 60, 5)
ecr_original = []
ecr_proposed = []
ecr_first = []
ecr_random_avg = []

for n in n_qubit_range:
    random.seed(42)
    np.random.seed(42)
    
    qc_i = create_multi_cx_qc(n, int(n * 0.8), seed=42)
    w_i, egi_i = build_edge_map(qc_i)
    
    # Original
    ecr_org = evaluate_cost(qc_i, backend)
    ecr_original.append(ecr_org)
    
    # Proposed (treewidth-based)
    res = find_optimal_cuts(qc_i, backend, max_cuts=1)
    ecr_proposed.append(res.ecr_after)
    
    # Naive: 最初の 2-qubit ゲートをカット
    all_idxs = [i for idxs in egi_i.values() for i in idxs]
    if all_idxs:
        qc_first = cut_gate_by_index(qc_i, min(all_idxs))
        ecr_first.append(evaluate_cost(qc_first, backend))
    else:
        ecr_first.append(ecr_org)
    
    # Random: ランダムにカット (3回平均)
    rand_costs = []
    for _ in range(3):
        if all_idxs:
            idx = random.choice(all_idxs)
            qc_rand = cut_gate_by_index(qc_i, idx)
            rand_costs.append(evaluate_cost(qc_rand, backend))
    ecr_random_avg.append(np.mean(rand_costs) if rand_costs else ecr_org)
    
    print(f"n={n:3d}: org={ecr_org}, proposed={res.ecr_after}, "
          f"first={ecr_first[-1]}, random={ecr_random_avg[-1]:.0f}")

In [ ]:
x = list(n_qubit_range)

plt.figure(figsize=(10, 5))
plt.plot(x, ecr_original, "o-", label="Original (no cut)")
plt.plot(x, ecr_proposed, "s-", label="Proposed (treewidth)")
plt.plot(x, ecr_first, "^-", label="Naive (first gate)")
plt.plot(x, ecr_random_avg, "d-", label="Random (avg of 3)")
plt.xlabel("Number of qubits")
plt.ylabel("ECR gates after transpile")
plt.title("ECR Gate Count: Proposed vs Baselines")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. K=2 カット

`max_cuts=2` で 2 箇所同時カットを行います。beam search で効率的に探索します。

In [ ]:
# 少し大きめの回路で K=2 を試す
qc2 = create_multi_cx_qc(n_qubit=40, n_cx=60, seed=7)

result_k1 = find_optimal_cuts(qc2, backend, max_cuts=1)
result_k2 = find_optimal_cuts(qc2, backend, max_cuts=2, beam_width=8)

print("=== K=1 (1箇所カット) ===")
print(f"  カットエッジ: {result_k1.cut_edges}")
print(f"  ECR: {result_k1.ecr_before} -> {result_k1.ecr_after} "
      f"(削減: {result_k1.ecr_reduction}, {result_k1.reduction_ratio:.1%})")

print(f"\n=== K=2 (2箇所カット) ===")
print(f"  カットエッジ: {result_k2.cut_edges}")
print(f"  ECR: {result_k2.ecr_before} -> {result_k2.ecr_after} "
      f"(削減: {result_k2.ecr_reduction}, {result_k2.reduction_ratio:.1%})")

## 6. パラメータの説明

`find_optimal_cuts` の主要パラメータ:

| パラメータ | デフォルト | 説明 |
|---|---|---|
| `max_cuts` | 1 | カット数 (1 or 2) |
| `M_candidates` | 30 | スコアリングで残す候補エッジ数 |
| `allowed_gate_names` | `("cx","cz","ecr","rzz")` | カット対象とするゲート名 |
| `gate_pick_policy` | `"first"` | 同一エッジの複数ゲートから選ぶポリシー (`"first"`, `"last"`, `"middle"`) |
| `seed_transpiler` | 42 | トランスパイルのシード |
| `optimization_level` | 1 | トランスパイルの最適化レベル |
| `beam_width` | 8 | K=2 の場合の beam 幅 |

## 7. 統合 API: `execute_subcircuits`

`DataClassSubQCParams`（機能2 の出力型）を受け取り、カット解析 → QPD/ベースライン実行 → `DataClassSubQCRes`（機能5 の入力型）を返す統合パイプラインです。

- ブレークイーブン判定 (`should_cut`) に基づいて自動的に QPD / ベースラインを切り替え
- 確率分布をカウント (`Dict[str, int]`) に変換して返す

In [ ]:
from qiskit import QuantumCircuit, qasm3
from circuit_preprocesser.treewidth_cut import (
    DataClassSubQCParams, DataClassSubQCRes, execute_subcircuits,
)

# ── テスト用の回路を QASM3 文字列として準備 ──
qc_a = QuantumCircuit(4)
qc_a.h(0)
for _ in range(3):
    for i in range(3): qc_a.cx(i, i + 1)

qc_b = QuantumCircuit(8)
for _ in range(3):
    for i in [0, 1, 2]: qc_b.cx(i, i + 1)
    for i in [4, 5, 6]: qc_b.cx(i, i + 1)
qc_b.cx(3, 4)  # ボトルネック辺

# ── DataClassSubQCParams を構築 ──
subqc_params_lst = [
    DataClassSubQCParams(
        subqc_id="chain_4q",
        subcircuit_role="time_left",
        assignment={"cut_0": 0},
        qasm3=qasm3.dumps(qc_a),
    ),
    DataClassSubQCParams(
        subqc_id="bottleneck_8q",
        subcircuit_role="time_right",
        assignment={"cut_0": 0},
        qasm3=qasm3.dumps(qc_b),
    ),
]

device_lst = [
    {"name": "ibm_sherbrooke", "fake_backend": "FakeSherbrooke", "gate_speed": 2e-9},
]

# ── 統合パイプライン実行 ──
results = execute_subcircuits(
    subqc_params_lst, device_lst,
    shots=8192, use_simulator=True, H0=1.0,
)

# ── 結果確認 ──
for r in results:
    top5 = sorted(r.counts.items(), key=lambda x: -x[1])[:5]
    print(f"{r.subqc_id} (role={r.subcircuit_role}, assignment={r.assignment}):")
    for bs, cnt in top5:
        print(f"  |{bs}⟩ : {cnt}")
    print(f"  total counts: {sum(r.counts.values())}")
    print()

## 8. QPD ゲートカットと分布再構成

`execute_qpd` は、指定したゲートを QPD（Quasi-Probability Decomposition）で 6 ブランチに分解し、
各ブランチの測定結果から元回路の確率分布を再構成します。

- CX/CZ 1 ゲートのカットで γ² = 9 のショットオーバーヘッド
- 中間測定ビットの符号補正を含む分布再構成

In [ ]:
from qiskit_aer import AerSimulator
from circuit_preprocesser.treewidth_cut import (
    execute_qpd, run_and_get_distribution, find_optimal_cuts,
)

# ── Bell 状態回路 (|00⟩ + |11⟩) / √2 ──
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)

sim = AerSimulator()

# ベースライン（カットなし）
dist_base = run_and_get_distribution(qc_bell, sim, 10000)
print("=== Baseline (no cut) ===")
for bs, p in sorted(dist_base.items(), key=lambda x: -x[1]):
    print(f"  |{bs}⟩ : {p:.4f}")

print()

# QPD カット（CX at gate_idx=1）
dist_qpd = execute_qpd(qc_bell, 1, sim, 30000)
print("=== QPD (cut CX, 30000 shots) ===")
for bs, p in sorted(dist_qpd.items(), key=lambda x: -abs(x[1])):
    sign = " " if p >= 0 else "-"
    print(f"  |{bs}⟩ : {sign}{abs(p):.4f}")
print(f"\n  合計: {sum(dist_qpd.values()):.4f}")
print(f"  負値エントリ: {sum(1 for v in dist_qpd.values() if v < 0)}")

In [ ]:
# ── より大きな回路での QPD: カット位置を自動選択 ──
qc_chain = QuantumCircuit(6)
qc_chain.h(0)
for _ in range(3):
    for i in range(5):
        qc_chain.cx(i, i + 1)

# TW2S でカット位置を選択
res = find_optimal_cuts(qc_chain, backend, max_cuts=1)
print(f"カット対象: edge={res.cut_edges[0]}, gate_idx={res.gate_indices[0]}, ΔECR={res.ecr_reduction}")
print()

# ベースライン vs QPD
dist_base = run_and_get_distribution(qc_chain, sim, 10000)
dist_qpd = execute_qpd(qc_chain, res.gate_indices[0], sim, 30000)

print("=== Baseline 上位5 ===")
for bs, p in sorted(dist_base.items(), key=lambda x: -x[1])[:5]:
    print(f"  |{bs}⟩ : {p:.4f}")

print("\n=== QPD 上位5 ===")
for bs, p in sorted(dist_qpd.items(), key=lambda x: -abs(x[1]))[:5]:
    sign = " " if p >= 0 else "-"
    print(f"  |{bs}⟩ : {sign}{abs(p):.4f}")
print(f"\n  QPD 合計確率: {sum(dist_qpd.values()):.4f}")